### Customer churn prediction for Interconnect

Interconnect, a telecom operator, aims to predict churn in order to proactivly retain customers who may be at risk of leaving. By targeting those in advance, the company can offer targeted promotions and service improvements.

The goal of this project is to analyze customer data and build machine learning model that can accurately predict weather custumer is likely to churn.

The dataset includes information about customer contracts, personal details, internet and phone services. Using this data, we will explore key factors that influence churn and develop models to predict it.

The main evaluation metric for this project is AUC-ROC with accuracy used as an additional metric.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_curve

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

from catboost import CatBoostClassifier

ModuleNotFoundError: No module named 'catboost'

In [ ]:
#loading datasets
contract = pd.read_csv('/datasets/final_provider/contract.csv')
personal = pd.read_csv('/datasets/final_provider/personal.csv')
internet = pd.read_csv('/datasets/final_provider/internet.csv')
phone = pd.read_csv('/datasets/final_provider/phone.csv')

In [ ]:
#merging datasets
df = contract.copy()

df = df.merge(personal, on='customerID', how='left')
df = df.merge(internet, on='customerID', how='left')
df = df.merge(phone, on='customerID', how='left')

In [ ]:
#filling in missing values
df = df.fillna('No')

In [ ]:
#converting TotalCharges datatype
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

#converting dates
df['BeginDate'] = pd.to_datetime(df['BeginDate'])
df['EndDate'] = pd.to_datetime(df['EndDate'], errors='coerce')


In [ ]:
#creating target
df['churn'] = df['EndDate'].notna().astype(int)

In [ ]:
#creating tenure
df['tenure'] = (pd.to_datetime('2020-02-01') - df['BeginDate']).dt.days

## Exploratory Data Analysis

In this section, we will explore key features to better understand data and key relationship with customer churn.

In [ ]:
#churn distribution 

df['churn'].value_counts().plot(kind='bar')
plt.title('Churn Distribution')
plt.xlabel('Churn (0 = No, 1 = Yes)')
plt.ylabel('Number of Customers')
plt.show()

The distribution shows that the majority of customers have not churned, while a smaller portion have left. This indicates that the dataset is moderately imbalanced, with more active customers than churned ones.

In [ ]:
#contract type vs churn

sns.countplot(x='Type', hue='churn', data=df)
plt.title('Contract type vs churn')
plt.show

The plot shows that the month to month customers are more likely to churn than those with longer term contracts.


In [ ]:
#montly charges vs churn

sns.boxplot(x='churn', y='MonthlyCharges', data=df)
plt.title('Monthly Charges by Churn')
plt.show()

This plot shows that the customers who leave the company generaly have higher monthley charges than those who stay.
This suggest that higher prices might be linked to higher risk of churn, possibly because customers feel they are not getting enough value for what they are paying.

In [ ]:
#tenure vs churn
sns.boxplot(x='churn', y='tenure', data=df)
plt.title('Tenure by churn')
plt.show()

The plot shows that the customers who leave usually have been with the company for shorter period of time.
in contrast, long term customers are more likely to stay. This suggest that the longer the customer stay, the less likely to churn.

In [ ]:
#dropping unnecessary columns
df_model = df.drop(columns=['customerID', 'BeginDate', 'EndDate'])

In [ ]:
#checking for duplicates after dropping columns
duplicates = df_model.duplicated().sum()
print('Number of duplicated rows:', duplicates)

In [ ]:
#removing duplicates
df_model = df_model.drop_duplicates()

In [ ]:
#separating features and target
features = df_model.drop('churn', axis=1)
target = df_model['churn']

In [ ]:
#splitting into training and test sets

#first split: train (60%) and temp (40%)
features_train, features_temp, target_train, target_temp = train_test_split(
    features,
    target,
    test_size=0.4,
    random_state=12345,
    stratify=target
)

#second split: validation (20%) and test (20%)
features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp,
    target_temp,
    test_size=0.5,
    random_state=12345,
    stratify=target_temp
)

In [ ]:
#checking shapes
print(features_train.shape)
print(features_valid.shape)
print(features_test.shape)


In [ ]:
#identifying numerical and categorical columns
categorical_features = features_train.select_dtypes(include='object').columns.tolist()
numeric_features = features_train.select_dtypes(exclude='object').columns.tolist()

print('Categorical features:', categorical_features)
print('Numeric features:', numeric_features)

In [ ]:
#building preprocessing pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

In [ ]:
#building dummy model

dummy_model = DummyClassifier(strategy='most_frequent')

dummy_model.fit(features_train, target_train)

dummy_pred = dummy_model.predict(features_valid)
dummy_proba = dummy_model.predict_proba(features_valid)[:, 1]

dummy_auc = roc_auc_score(target_valid, dummy_proba)
dummy_acc = accuracy_score(target_valid, dummy_pred)

print('Dummy Model AUC-ROC:', dummy_auc)
print('Dummy Model Accuracy:', dummy_acc)

### Baseline Model (Dummy Classifier)

We used a dummy classifier as a baseline model, predicting the majority class.

This model provides a minimum performance benchmark. All real models should outperform this baseline to confirm that they are learning meaningful patterns from the data.

In [ ]:
#building logistic regression pipeline

lr_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=12345))
])

In [ ]:
lr_model.fit(features_train, target_train)

In [ ]:
#evaluating on validation set

lr_valid_pred = lr_model.predict(features_valid)
lr_valid_proba = lr_model.predict_proba(features_valid)[:, 1]

lr_valid_auc = roc_auc_score(target_valid, lr_valid_proba)
lr_valid_acc = accuracy_score(target_valid, lr_valid_pred)

print('Logistic Regression Validation AUC-ROC:', lr_valid_auc)
print('Logistic Regression Validation Accuracy:', lr_valid_acc)

### Logistic Regression model

We trained on the training set and evaluated on the validation set. We achieved an AUC-ROC score of 0.843 and an accuracy of 0.804.


In [ ]:
#moving on to a stronger model, Random Forest
 
#building pipeline
rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=200,
        random_state=12345
    ))
])

#training on training set
rf_model.fit(features_train, target_train)

#evaluating on validation set
rf_valid_pred = rf_model.predict(features_valid)
rf_valid_proba = rf_model.predict_proba(features_valid)[:, 1]

#calculating metrics
rf_valid_auc = roc_auc_score(target_valid, rf_valid_proba)
rf_valid_acc = accuracy_score(target_valid, rf_valid_pred)

print('Random Forest Validation AUC-ROC:', rf_valid_auc)
print('Random Forest Validation Accuracy:', rf_valid_acc)


### Random Forest Model

We trained Random Forest model on the training set and evaluated on the validation set.

We achieved an AUC-ROC score of 0.868 and an accuracy of 0.833, outperforming Logistic Regression. This indicates that Random Forest is better at capturing non-linear relationships in the data.

In [ ]:
#trying to push for even better results with CatBoost

#initializing model
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    eval_metric='AUC',
    random_seed=12345,
    verbose=0
)

#training on training set
cat_model.fit(
    features_train,
    target_train,
    cat_features=categorical_features
)

#evaluating on validation set
cat_valid_pred = cat_model.predict(features_valid)
cat_valid_proba = cat_model.predict_proba(features_valid)[:, 1]

#calculating metrics
cat_valid_auc = roc_auc_score(target_valid, cat_valid_proba)
cat_valid_acc = accuracy_score(target_valid, cat_valid_pred)

print('CatBoost Validation AUC-ROC:', cat_valid_auc)
print('CatBoost Validation Accuracy:', cat_valid_acc)


### Model Comparison

We evalueted a baseline Dummy model and three machine learning models: Logistic Regression, Random Forest, and CatBoost.

The Dummy model achieved an AUC-ROC score of 0.50, serving as a baseline. All trained models significantly outperformed this baseline.

CatBoost achieved the best performance, with an AUC-ROC score of 0.925 and an accuracy of 0.892 on the validation set, and we selected it as the final model.

In [ ]:
#final test evaluation 

cat_test_pred = cat_model.predict(features_test)
cat_test_proba = cat_model.predict_proba(features_test)[:, 1]

final_auc = roc_auc_score(target_test, cat_test_proba)
final_acc = accuracy_score(target_test, cat_test_pred)

print('Final Test AUC-ROC:', final_auc)
print('Final Test Accuracy:', final_acc)

### Final Model Evaluation

This model achieved an AUC-ROC score of 0.908 and an accuracy of 0.878. These results confirm that the model performs well on unseen data and is effective at predicting customer churn.

In [ ]:
#AUC-ROC curve display

fpr, tpr, thresholds = roc_curve(target_test, cat_test_proba)

plt.plot(fpr, tpr, label='CatBoost (AUC = {:.3f})'.format(final_auc))
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

### ROC Curve

The ROC curve shows how well the model separates customers who churn from those who stay.

The closer the curve is to the top-left corner, the better the model performs.

### Conclusion

In this project, we built a model to predict customer churn for Interconnect using customer, contract, and service data.

After comparing several models, CatBoost performed the best and achieved strong results on both the validation and test sets. This shows that the model is able to accurately identify customers who are at risk of leaving.

Key factors influencing churn include contract type, tenure, monthly charges, and access to services like tech support.

Overall, the model can help the company better understand customer behavior and take proactive steps to retain high-risk customers.


